<a href="https://colab.research.google.com/github/UMAIR-777/llm-finetune-lab/blob/main/Fine_tuned__Qwen_1_5_b_with_soup%20framework_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



###  Check GPU Allocation & Memory (`!nvidia-smi`)
Runs the NVIDIA System Management Interface in the Colab Linux terminal to verify our hardware setup:
* **GPU Model:** Confirms a GPU is attached (e.g., Tesla T4, L4, A100) instead of just a CPU.
* **VRAM Usage:** Shows total and used video memory (e.g., `0MiB / 15360MiB`).
* **CUDA Version:** Displays the driver and CUDA version for PyTorch/TensorFlow compatibility.

> **Troubleshooting:** If this fails with a driver communication error, go to **Runtime > Change runtime type** and select a **GPU**.



In [ ]:
!nvidia-smi

Wed Nov 11 13:51:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             48 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   AuthenticAMD
  Model name:                AMD EPYC 7B12
    CPU family:              23
    Model:                   49
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4499.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ht s
                             yscall nx mmxext fxsr_opt pdpe1gb rdtscp lm constan
                             t_tsc rep_good nopl nonstop_tsc cpuid extd_apicid t
                             sc_known_freq pni pclmulqdq ssse3 fma cx16 sse4_1 s
                             se4_2 

In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       9.1Gi       2.2Mi       2.7Gi        11Gi
Swap:             0B          0B          0B


In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   85M  108G   1% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.3G  710M  65% /usr/sbin/docker-init
tmpfs           6.4G   64K  6.4G   1% /var/colab
/dev/sda1       114G  925M  114G   1% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())

PyTorch version: 2.14.1+cu130
CUDA available: True
PyTorch CUDA version: 13.0
GPU: Tesla T4
CUDA device count: 1


In [ ]:
!pip -q install transformers huggingface_hub

In [ ]:
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

config = AutoConfig.from_pretrained(MODEL_ID)

print("Architecture:", config.architectures)
print("Model type:", config.model_type)
print("Hidden size:", config.hidden_size)
print("Layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", config.num_key_value_heads)
print("Intermediate size:", config.intermediate_size)
print("Vocabulary size:", config.vocab_size)
print("Maximum positions:", config.max_position_embeddings)
print("Tied embeddings:", config.tie_word_embeddings)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

Architecture: ['Qwen2ForCausalLM']
Model type: qwen2
Hidden size: 1536
Layers: 28
Attention heads: 12
KV heads: 2
Intermediate size: 8960
Vocabulary size: 151936
Maximum positions: 32768
Tied embeddings: True


In [ ]:

# Qwen2.5-1.5B architecture
L = 28
H = 1536
N_Q = 12
N_KV = 2
I = 8960
V = 151936

# Architecture settings
ATTENTION_BIAS = True
MLP_BIAS = False
TIED_EMBEDDINGS = True

# 1. Derive head dimension
D = H // N_Q
KV_DIM = N_KV * D

# 2. Attention weights
q_weights = H * H
k_weights = H * KV_DIM
v_weights = H * KV_DIM
o_weights = H * H

attention_weights = (
    q_weights + k_weights + v_weights + o_weights
)

# 3. Attention biases
attention_biases = (
    H + KV_DIM + KV_DIM + 0
    if ATTENTION_BIAS else 0
)

attention_total = attention_weights + attention_biases

# 4. Gated MLP (SwiGLU)
gate_weights = H * I
up_weights = H * I
down_weights = I * H

mlp_weights = gate_weights + up_weights + down_weights
mlp_biases = 0  # Qwen2 mlp_bias=False
mlp_total = mlp_weights + mlp_biases

# 5. RMSNorm parameters
block_norms = 2 * H

# 6. One Transformer block
block_total = attention_total + mlp_total + block_norms

# 7. All Transformer blocks
all_blocks = L * block_total

# 8. Embeddings and final norm
embedding = V * H
final_norm = H

lm_head = 0 if TIED_EMBEDDINGS else V * H

# 9. Total
total_parameters = all_blocks + embedding + final_norm + lm_head

# Display breakdown
print("Head dimension:", D)
print("KV dimension:", KV_DIM)
print("Attention per layer:", attention_total)
print("MLP per layer:", mlp_total)
print("Norms per layer:", block_norms)
print("One Transformer block:", block_total)
print("All Transformer blocks:", all_blocks)
print("Embedding:", embedding)
print("Final norm:", final_norm)
print("LM head:", lm_head)
print("TOTAL PARAMETERS:", total_parameters)


Head dimension: 128
KV dimension: 256
Attention per layer: 5507072
MLP per layer: 41287680
Norms per layer: 3072
One Transformer block: 46797824
All Transformer blocks: 1310339072
Embedding: 233373696
Final norm: 1536
LM head: 0
TOTAL PARAMETERS: 1543714304


In [ ]:

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="cpu"
)

actual_total = sum(p.numel() for p in model.parameters())

print("Actual parameter count:", actual_total)
print("Manual parameter count:", total_parameters)
print("Difference:", actual_total - total_parameters)
print(
    "Match:",
    actual_total == total_parameters
)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Actual parameter count: 1543714304
Manual parameter count: 1543714304
Difference: 0
Match: True


In [ ]:
import torch

# Inspect parameter dtypes
from collections import Counter

dtype_counts = Counter(
    (str(p.dtype), p.numel())
    for p in model.parameters()
)

for dtype, count in dtype_counts:
    print(dtype, count)

# Calculate actual parameter storage
actual_weight_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

print("\nTotal weight memory:")
print("Bytes:", actual_weight_bytes)
print("GiB:", actual_weight_bytes / (1024**3))

torch.bfloat16 233373696
torch.bfloat16 2359296
torch.bfloat16 1536
torch.bfloat16 393216
torch.bfloat16 256
torch.bfloat16 13762560

Total weight memory:
Bytes: 3087428608
GiB: 2.875391960144043


In [ ]:
B = 1
S = 512
V = 151936
bytes_per_element = 2

logits_bytes = B * S * V * bytes_per_element

print("Bytes:", logits_bytes)
print("MiB:", logits_bytes / (1024**2))

Bytes: 155582464
MiB: 148.375


In [ ]:

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Main project directory
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/qwen-tool-calling-finetuning"
)

# Project folders
folders = [
    "data/raw",
    "data/merged",
    "data/processed",
    "data/validation",
    "preprocessing",
    "evaluation",
    "checkpoints",
    "output",
    "notebooks"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project created successfully!")
print("Location:", PROJECT_ROOT)


Mounted at /content/drive
Project created successfully!
Location: /content/drive/MyDrive/qwen-tool-calling-finetuning


In [ ]:

from pathlib import Path

PREPROCESSING_DIR = PROJECT_ROOT / "preprocessing"

# 1. Merge datasets without modifying source records
merge_script = r'''
import json
from pathlib import Path
from copy import deepcopy


def load_json_records(file_path):
    """Load a JSON array or JSONL dataset."""
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as f:
        if path.suffix.lower() == ".jsonl":
            return [json.loads(line) for line in f if line.strip()]

        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        return [data]

    raise ValueError("Unsupported dataset structure")


def merge_datasets(original_records, custom_records):
    """Merge records while preserving their original structures."""
    merged = deepcopy(original_records) + deepcopy(custom_records)

    ids = [
        record.get("id")
        for record in merged
        if record.get("id") is not None
    ]

    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate IDs detected. Resolve them first.")

    return merged


def save_json(records, output_path):
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    print("Dataset merge module is ready.")
'''

# 2. Conversion placeholder
convert_script = r'''
def convert_record(record):
    """
    Convert one source record into Soup tool-calling format.

    Conversion rules will be implemented after inspecting
    the actual xLAM and custom dataset schemas.
    """
    raise NotImplementedError(
        "Inspect source schemas before implementing conversion."
    )


if __name__ == "__main__":
    print("Conversion module created. Awaiting schema inspection.")
'''

# 3. Validation starter
validate_script = r'''
import json
from pathlib import Path


def validate_jsonl(file_path):
    """Check JSONL syntax and basic message structure."""
    errors = []
    total = 0

    with Path(file_path).open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            total += 1

            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append({
                    "line": line_number,
                    "error": str(e)
                })
                continue

            if not isinstance(record.get("messages"), list):
                errors.append({
                    "line": line_number,
                    "error": "Missing messages list"
                })

            if not isinstance(record.get("tools"), list):
                errors.append({
                    "line": line_number,
                    "error": "Missing tools list"
                })

    return {
        "total_records": total,
        "error_count": len(errors),
        "errors": errors
    }


if __name__ == "__main__":
    print("Validation module is ready.")
'''

scripts = {
    "merge_datasets.py": merge_script,
    "convert_to_tool_calling.py": convert_script,
    "validate_dataset.py": validate_script,
}

for filename, content in scripts.items():
    (PREPROCESSING_DIR / filename).write_text(
        content.strip() + "\n",
        encoding="utf-8"
    )

print("Created preprocessing scripts:")
for filename in scripts:
    print("-", filename)


Created preprocessing scripts:
- merge_datasets.py
- convert_to_tool_calling.py
- validate_dataset.py


In [ ]:

readme = """
# Qwen Tool Calling Fine-Tuning

## Objective
Fine-tune Qwen2.5-1.5B-Instruct to improve tool-calling
decisions, including clarification behavior.

## Dataset categories
- clear_intent_direct_call
- vague_but_inferable_direct_call
- ambiguous_intent_clarification
- missing_required_parameters_clarification
- multi_turn_clarification_to_tool_call

## Pipeline
1. Store original datasets in data/raw/
2. Merge source records.
3. Convert to Soup tool-calling format.
4. Validate the processed JSONL.
5. Configure and run Soup training.
6. Evaluate model behavior.

## Important
Raw source datasets must remain unchanged.
"""

(PROJECT_ROOT / "README.md").write_text(
    readme.strip() + "\n",
    encoding="utf-8"
)

print("README created.")


README created.


In [ ]:

def show_tree(directory, prefix=""):
    items = sorted(directory.iterdir(), key=lambda p: (p.is_file(), p.name))

    for index, item in enumerate(items):
        last = index == len(items) - 1
        connector = "└── " if last else "├── "

        print(prefix + connector + item.name)

        if item.is_dir():
            extension = "    " if last else "│   "
            show_tree(item, prefix + extension)


print(PROJECT_ROOT.name + "/")
show_tree(PROJECT_ROOT)


qwen-tool-calling-finetuning/
├── checkpoints
├── data
│   ├── merged
│   ├── processed
│   ├── raw
│   └── validation
├── evaluation
├── notebooks
├── output
├── preprocessing
│   ├── convert_to_tool_calling.py
│   ├── merge_datasets.py
│   └── validate_dataset.py
└── README.md


In [7]:
%run /content/drive/MyDrive/qwen-tool-calling-finetuning/preprocessing/merge_datasets.py

[RAW] Saved 700 records -> /content/drive/MyDrive/qwen-tool-calling-finetuning/data/raw/train_clean_700.json
[RAW] Saved 300 records -> /content/drive/MyDrive/qwen-tool-calling-finetuning/data/raw/train_to_inject_300.json
[RAW] Saved 200 records -> /content/drive/MyDrive/qwen-tool-calling-finetuning/data/raw/test_200.json


In [3]:
%run /content/drive/MyDrive/qwen-tool-calling-finetuning/preprocessing/data_augmentation.py

Already completed: 0 / 300
Remaining pending: 300
Processing next 15 new records (paced at 13s/req for 5 RPM limit)...

[1/15] Saved id=11405 (Total saved: 1/300)
[2/15] Saved id=40338 (Total saved: 2/300)
[3/15] Saved id=37296 (Total saved: 3/300)
[4/15] Saved id=19723 (Total saved: 4/300)
[5/15] Saved id=26612 (Total saved: 5/300)
[6/15] Saved id=35909 (Total saved: 6/300)
[7/15] Saved id=54639 (Total saved: 7/300)
[8/15] Saved id=26 (Total saved: 8/300)
[9/15] Saved id=19914 (Total saved: 9/300)
[10/15] Saved id=18803 (Total saved: 10/300)
[11/15] Saved id=13774 (Total saved: 11/300)
[12/15] Saved id=28173 (Total saved: 12/300)
[13/15] Saved id=51495 (Total saved: 13/300)
[14/15] Saved id=38009 (Total saved: 14/300)
[15/15] Saved id=39758 (Total saved: 15/300)

Batch complete! Total unique records in /content/drive/MyDrive/qwen-tool-calling-finetuning/data/raw/train_injected_300.json: 15


In [1]:
%run /content/drive/MyDrive/qwen-tool-calling-finetuning/preprocessing/convert_to_tool_calling.py

Original records (query style): 700
New records (turns style): 300
Saved merged source: /content/drive/MyDrive/qwen-tool-calling-finetuning/data/merged/merged_source.json
Total source records: 1000

Conversion successful.
Converted records: 1000
Training file: /content/drive/MyDrive/qwen-tool-calling-finetuning/data/merged/merged_soup.jsonl
Metadata file: /content/drive/MyDrive/qwen-tool-calling-finetuning/data/merged/merged_metadata.json
